# 02 Grid, Network Preparation, and Four-Mode Isochrones

This notebook mirrors the supplied Shanghai network-extract notebook more closely than the short version did. It first prepares road travel times, then creates a 500 m analysis grid, then computes the practical four-mode accessibility surfaces used in the final score.


# Setup


In [ ]:
from pathlib import Path
import sys
import itertools
from operator import itemgetter
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / ".deps"))
sys.path.insert(0, str(ROOT))

import geopandas as gpd
from shapely.geometry import Point, LineString
from scipy.spatial import cKDTree

from scripts.build_outputs import (
    PROJECT_CRS,
    MODE_RADII_M,
    add_accessibility_scores,
    add_districts,
    add_h3_ids,
    add_housing_price,
    create_500m_grid,
    load_env,
    load_poi_points,
    load_supporting_points,
    locate_paths,
    merge_groups,
    read_file,
    union_geometry,
)

load_env()
paths = locate_paths()


# I. Road-network preparation


In [ ]:
roads = gpd.read_parquet(paths["roads"])
roads = roads.set_crs(PROJECT_CRS, allow_override=True) if roads.crs is None else roads.to_crs(PROJECT_CRS)
roads["length_m"] = roads.geometry.length
roads[["osm_id", "highway", "level", "bicycle", "foot", "geometry"]].head(3)


In [ ]:
# Speeds mirror the supplied shanghai-network-extract notebook.
SPEEDS_MPS = {
    "foot": 1.33,
    "bike": 3.05,
    "car": 8.30,
}

roads["foot_time_s"] = roads["length_m"] / SPEEDS_MPS["foot"]
roads["bike_time_s"] = roads["length_m"] / SPEEDS_MPS["bike"]
roads["car_time_s"] = roads["length_m"] / SPEEDS_MPS["car"]

roads[["length_m", "foot_time_s", "bike_time_s", "car_time_s"]].describe().round(2)


In [ ]:
gdf_car = roads[pd.to_numeric(roads["level"], errors="coerce").fillna(0) >= 3].copy()
gdf_foot = roads[pd.to_numeric(roads["foot"], errors="coerce").fillna(0) > 0].copy()
gdf_bike = roads[pd.to_numeric(roads["bicycle"], errors="coerce").fillna(0) > 0].copy()

pd.DataFrame([
    {"mode": "foot", "edges": len(gdf_foot), "km": gdf_foot["length_m"].sum() / 1000},
    {"mode": "bike", "edges": len(gdf_bike), "km": gdf_bike["length_m"].sum() / 1000},
    {"mode": "car", "edges": len(gdf_car), "km": gdf_car["length_m"].sum() / 1000},
]).round(1)


## I.1. Nearest-edge helper, following the reference notebook


In [ ]:
def ckdnearest(gdfA, gdfB):
    # Return nearest row indices in gdfB for every coordinate in gdfA.
    A = np.concatenate([np.array(geom.coords) for geom in gdfA.geometry.to_list()])
    B_parts = [np.array(geom.coords) for geom in gdfB.geometry.to_list()]
    B_ix = tuple(itertools.chain.from_iterable([
        itertools.repeat(i, x) for i, x in enumerate(list(map(len, B_parts)))
    ]))
    B = np.concatenate(B_parts)
    ckd_tree = cKDTree(B)
    dist, idx = ckd_tree.query(A, k=1)
    idx = np.atleast_1d(idx)
    row_positions = np.array([B_ix[int(i)] for i in idx], dtype=int)
    return row_positions, np.atleast_1d(dist)


In [ ]:
sample_origin = gpd.GeoDataFrame(
    [["People Square", Point(21351908, 3456756)]],
    columns=["name", "geometry"],
    crs=PROJECT_CRS,
)

idx_car, dist_car = ckdnearest(sample_origin, gdf_car)
idx_bike, dist_bike = ckdnearest(sample_origin, gdf_bike)
idx_foot, dist_foot = ckdnearest(sample_origin, gdf_foot)

pd.DataFrame([
    {"mode": "car", "edge_id": gdf_car.iloc[idx_car].index[0], "nearest_m": float(dist_car[0])},
    {"mode": "bike", "edge_id": gdf_bike.iloc[idx_bike].index[0], "nearest_m": float(dist_bike[0])},
    {"mode": "foot", "edge_id": gdf_foot.iloc[idx_foot].index[0], "nearest_m": float(dist_foot[0])},
]).round(2)


## I.2. Dijkstra isochrone pattern


In [ ]:
DIJKSTRA_PSEUDOCODE = "\n".join([
    "1. Filter the road network by mode: foot, bike, or car.",
    "2. Convert each road edge to graph edges with travel-time weight.",
    "3. Snap an origin point to the nearest eligible edge.",
    "4. Run Dijkstra until travel time exceeds 15 * 60 seconds.",
    "5. Keep fully reachable edges and cut partially reachable edge segments.",
    "6. Buffer/union reachable segments into an isochrone polygon.",
])

print(DIJKSTRA_PSEUDOCODE)


In [ ]:
threshold_s = 15 * 60
euclidean_buffers = gpd.GeoDataFrame(
    [
        ["walk", sample_origin.geometry.iloc[0].buffer(threshold_s * SPEEDS_MPS["foot"])],
        ["bike", sample_origin.geometry.iloc[0].buffer(threshold_s * SPEEDS_MPS["bike"])],
        ["car", sample_origin.geometry.iloc[0].buffer(threshold_s * SPEEDS_MPS["car"])],
    ],
    columns=["mode", "geometry"],
    crs=PROJECT_CRS,
)
euclidean_buffers["area_km2"] = euclidean_buffers.geometry.area / 1_000_000
euclidean_buffers[["mode", "area_km2"]].round(2)


# II. Build the 500 m grid


In [ ]:
city = read_file(paths["city_boundary"], crs=PROJECT_CRS)
built = read_file(paths["built_area_2020"], crs=PROJECT_CRS)
city_union = union_geometry(city)
built["geometry"] = built.geometry.make_valid().intersection(city_union)
built = built[~built.geometry.is_empty].copy()

pd.Series({
    "city_area_km2": city.geometry.area.sum() / 1_000_000,
    "built_area_km2": built.geometry.area.sum() / 1_000_000,
    "built_polygons": len(built),
}).round(2)


In [ ]:
grid = create_500m_grid(built)
grid = add_districts(grid, paths["district_boundary"])

grid[["grid_id", "district", "cell_area_m2", "centroid_x", "centroid_y"]].head()


In [ ]:
grid.groupby("district").size().sort_values(ascending=False).rename("grid_cells").head(20).to_frame()


# III. Load amenities and supporting layers


In [ ]:
poi_groups, poi_meta = load_poi_points()
support_groups, support_meta = load_supporting_points(paths)
groups = merge_groups(poi_groups, support_groups)

group_counts = pd.Series({k: len(v) for k, v in groups.items()}).sort_values(ascending=False)
print("POI status:", poi_meta["poi_status"])
print("POI rows loaded:", f"{poi_meta['poi_rows_loaded']:,}")
group_counts.to_frame("features")


In [ ]:
support_meta


# IV. Four-mode accessibility scores


In [ ]:
pd.DataFrame(
    [{"mode": mode, "radius_m": radius, "radius_km": radius / 1000} for mode, radius in MODE_RADII_M.items()]
)


In [ ]:
grid = add_accessibility_scores(grid, groups)
grid = add_housing_price(grid, groups["housing_price"])
grid = add_h3_ids(grid)

score_cols = [
    "baseline_walk", "health_walk", "composite_walk",
    "composite_bike", "composite_transit", "composite_car",
    "metro_dist_m", "major_road_dist_m", "rent_band",
]
grid[["grid_id", "district"] + score_cols].head()


In [ ]:
mode_summary = grid[[
    "composite_walk", "composite_bike", "composite_transit", "composite_car",
    "baseline_walk", "health_walk",
]].describe().round(2)
mode_summary


In [ ]:
category_cols = [
    "food_walk_score", "healthcare_walk_score", "education_walk_score",
    "open_space_walk_score", "transit_access_walk_score", "civic_walk_score",
    "sport_walk_score", "green_outdoor_walk_score", "cycling_env_walk_score",
    "healthy_food_walk_score", "env_quality_walk_score",
]
grid[category_cols].mean().sort_values(ascending=False).round(2).to_frame("mean_score")


# V. Cache the grid layer


In [ ]:
grid_out = ROOT / "data" / "processed" / "shanghai_500m_health_grid.parquet"
grid.to_parquet(grid_out, index=False)
print(f"Wrote {grid_out}")
print(f"Rows: {len(grid):,}")
print(f"Columns: {len(grid.columns):,}")


In [ ]:
qa = pd.DataFrame([
    {"check": "grid has H3 IDs", "value": grid["h3_id"].notna().all()},
    {"check": "all composite walk scores are finite", "value": np.isfinite(grid["composite_walk"]).all()},
    {"check": "metro distances computed", "value": grid["metro_dist_m"].notna().mean() > 0.95},
    {"check": "district label coverage", "value": grid["district"].ne("Unknown").mean()},
])
qa


## Notebook output

This notebook produces `data/processed/shanghai_500m_health_grid.parquet`. The final notebook aggregates that cell-level layer into H3 resolution 8 for the web application.
